# TechCorp Challenge IA — Medical Data Pipeline
**Roles:** DATA + IA  
**Covers:** Phases 0–4 (Environment → EDA → Cleaning → Formatting → Quality Report + Export)  
**Output:** Cleaned parquet, chat-formatted jsonl, quality report  
**Dataset:** `ruslanmv/ai-medical-chatbot` (~257k rows, columns: Description / Patient / Doctor)

---
## ⚠️ Before running
1. Runtime → Change runtime type → **GPU** (T4 minimum, A100 preferred)
2. Run cells **top to bottom in order**
3. Phase 0 will restart the kernel once — that is expected, re-run from Phase 1 onward after restart

---
## Phase 0 — Environment & De-risking
**Why first:** Dependency conflicts are the #1 time sink. Unsloth pins a compatible torch/transformers/bitsandbytes stack for us. Mounting Drive means a Colab disconnect won't kill our checkpoint.

In [ ]:
# 0.1 — Install Unsloth + supporting libraries
# Unsloth resolves the torch/bitsandbytes/transformers triangle automatically.
# datasketch → near-duplicate detection in Phase 2
# langdetect → language filtering in Phase 2
# rouge-score / bert-score → evaluation metrics in Phase 7 (Training notebook)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes datasets
!pip install -q datasketch langdetect rouge-score bert-score
print("Installation complete — kernel restart expected on first run.")

In [ ]:
# 0.2 — GPU check
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print("GPU:", result.stdout.strip() if result.returncode == 0 else "No GPU detected — switch runtime type!")

In [ ]:
# 0.3 — Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/TechCorp_Challenge'
os.makedirs(f'{DRIVE_BASE}/data', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/reports', exist_ok=True)
print(f"Drive mounted. Output directory: {DRIVE_BASE}")

In [ ]:
# 0.4 — Core imports
import pandas as pd
import numpy as np
import re
import json
import hashlib
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
TARGET_ROWS = 15_000   # Phase 2 — final curated size
MAX_SEQ_LENGTH = 2048  # Phase 3 — token budget per example

print("Imports OK. Target dataset size:", TARGET_ROWS)

---
## Phase 1 — Data Load & EDA
**Why:** You cannot define cleaning rules for data you haven't seen. Two key findings drive the whole pipeline:
- **Token-length distribution** → sets `max_seq_length` so we don't blow GPU memory
- **Boilerplate patterns** → defines the regex strip list for Phase 2

We also baseline exact-duplicate rate, language mix, and null counts here.

In [ ]:
# 1.1 — Load from HuggingFace Hub
print("Loading dataset from HuggingFace Hub (first run may take ~1 min) ...")
df_raw = pd.read_parquet("hf://datasets/ruslanmv/ai-medical-chatbot/dialogues.parquet")

print(f"\nShape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"\nDtypes:\n{df_raw.dtypes}")
df_raw.head(3)

In [ ]:
# 1.2 — Null / missing analysis
null_counts = df_raw.isna().sum()
null_pct = (null_counts / len(df_raw) * 100).round(2)
print("Null counts and % per column:")
print(pd.concat([null_counts, null_pct], axis=1, keys=['nulls', '%']))

In [ ]:
# 1.3 — Char-length distributions
df_raw['pat_char_len'] = df_raw['Patient'].fillna('').str.len()
df_raw['doc_char_len'] = df_raw['Doctor'].fillna('').str.len()

print("Patient message lengths (chars):")
print(df_raw['pat_char_len'].describe().round(1))
print("\nDoctor answer lengths (chars):")
print(df_raw['doc_char_len'].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col, label in zip(axes,
                           ['pat_char_len', 'doc_char_len'],
                           ['Patient (chars)', 'Doctor (chars)']):
    clipped = df_raw[col].clip(upper=df_raw[col].quantile(0.99))
    ax.hist(clipped, bins=60, color='steelblue', edgecolor='white', linewidth=0.4)
    ax.set_title(f'{label} distribution (99th pct clip)')
    ax.set_xlabel('Characters'); ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/reports/char_length_distributions.png', dpi=120)
plt.show()

In [ ]:
# 1.4 — Token-length estimation using the base model tokenizer
# We sample 5k rows so this is fast; result sets max_seq_length for training.
from transformers import AutoTokenizer

print("Loading tokenizer for Phi-3.5-mini-instruct ...")
tok = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct", trust_remote_code=True)

sample_5k = df_raw.dropna(subset=['Patient','Doctor']).sample(5000, random_state=SEED)

def count_tokens(patient_text, doctor_text):
    combined = f"USER: {patient_text}\nASSISTANT: {doctor_text}"
    return len(tok.encode(combined, truncation=False))

print("Counting tokens on 5k sample ...")
sample_5k = sample_5k.copy()
sample_5k['token_len'] = [
    count_tokens(p, d)
    for p, d in zip(sample_5k['Patient'], sample_5k['Doctor'])
]

print("\nToken length distribution (5k sample):")
print(sample_5k['token_len'].describe().round(1))

pcts = [50, 75, 90, 95, 99]
print("\nPercentile breakdown:")
for p in pcts:
    print(f"  p{p}: {int(np.percentile(sample_5k['token_len'], p))} tokens")

fig, ax = plt.subplots(figsize=(10, 4))
clipped = sample_5k['token_len'].clip(upper=sample_5k['token_len'].quantile(0.99))
ax.hist(clipped, bins=60, color='darkorange', edgecolor='white', linewidth=0.4)
ax.axvline(MAX_SEQ_LENGTH, color='red', linestyle='--', label=f'max_seq_length={MAX_SEQ_LENGTH}')
ax.set_title('Token length distribution (5k sample, 99th pct clip)')
ax.set_xlabel('Tokens'); ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/reports/token_length_distribution.png', dpi=120)
plt.show()

In [ ]:
# 1.5 — Exact duplicate rate
n_exact_dups = df_raw.duplicated(subset=['Patient', 'Doctor']).sum()
print(f"Exact duplicates (Patient+Doctor): {n_exact_dups:,} ({n_exact_dups/len(df_raw)*100:.1f}% of rows)")

In [ ]:
# 1.6 — Language detection on a 3k sample
from langdetect import detect, LangDetectException

def safe_detect(text):
    try:
        return detect(str(text)[:500])  # first 500 chars is enough
    except LangDetectException:
        return 'unknown'

sample_lang = df_raw['Doctor'].dropna().sample(3000, random_state=SEED)
print("Detecting languages on 3k Doctor-answer sample ...")
langs = sample_lang.apply(safe_detect)
lang_counts = langs.value_counts()
print("\nTop 10 detected languages:")
print(lang_counts.head(10))

In [ ]:
# 1.7 — Manual sample: read 30 (Patient, Doctor) pairs to spot boilerplate
sample_view = df_raw.dropna(subset=['Patient','Doctor']).sample(30, random_state=SEED)
for i, row in enumerate(sample_view.itertuples(), 1):
    print(f"--- Example {i} ---")
    print(f"PATIENT: {row.Patient[:300]}")
    print(f"DOCTOR:  {row.Doctor[:400]}")
    print()

---
## Phase 2 — Cleaning & Quality Filtering
**Why:** For LoRA, quality beats volume. We apply filters in a deliberate order and log row counts at each step — these counts ARE the data quality report.

**Order matters:** strip HTML before length-filtering (HTML inflates char count); dedup after stripping (boilerplate creates near-dups); language-filter late (langdetect is slow, run on a smaller set).

In [ ]:
# 2.0 — Audit log: tracks row count after each step
audit = []

def log_step(label, df):
    entry = {'step': label, 'rows': len(df)}
    audit.append(entry)
    print(f"  [{label}] {len(df):>7,} rows remaining")
    return df

df = df_raw[['Patient', 'Doctor']].copy()
log_step('0 — raw load', df)

In [ ]:
# 2.1 — Drop null / empty Patient or Doctor
df = df.dropna(subset=['Patient', 'Doctor'])
df = df[df['Patient'].str.strip().str.len() > 0]
df = df[df['Doctor'].str.strip().str.len() > 0]
log_step('1 — drop null/empty', df)

In [ ]:
# 2.2 — Strip HTML tags and collapse whitespace
# Medical scrapes often contain <br>, <p>, &nbsp; etc.
HTML_TAG = re.compile(r'<[^>]+>')
NBSP = re.compile(r'&nbsp;|&#160;')
MULTI_SPACE = re.compile(r' {2,}')
MULTI_NEWLINE = re.compile(r'\n{3,}')

def clean_html(text):
    text = NBSP.sub(' ', text)
    text = HTML_TAG.sub(' ', text)
    text = MULTI_SPACE.sub(' ', text)
    text = MULTI_NEWLINE.sub('\n\n', text)
    return text.strip()

df['Patient'] = df['Patient'].apply(clean_html)
df['Doctor'] = df['Doctor'].apply(clean_html)
log_step('2 — strip HTML', df)

In [ ]:
# 2.3 — Remove boilerplate (patterns identified in Phase 1 EDA)
# Leading greetings in Patient messages
PAT_LEADING = re.compile(
    r'^(hi\s*[,.]?\s*|hello\s*[,.]?\s*|dear doctor\s*[,.]?\s*|'  
    r'dear dr\.?\s*[,.]?\s*|good (morning|afternoon|evening)\s*[,.]?\s*)',
    re.IGNORECASE
)
# Trailing signatures in Doctor answers
DOC_TRAILING = re.compile(
    r'(thanks\s+for\s+(your\s+)?query\.?|'  
    r'hope\s+this\s+helps?\.?|'  
    r'take\s+care\.?|'  
    r'get\s+well\s+soon\.?|'  
    r'(kind\s+)?regards?,?.*|'  
    r'do\s+consult\s+your\s+doctor.*|'  
    r'welcome\s+to\s+(health|hcm|icliniq).*)'  
    r'[\s\S]{0,200}$',
    re.IGNORECASE
)
# Welcome lines at the start of Doctor answers
DOC_LEADING_WELCOME = re.compile(
    r'^(welcome\s+to\s+(health\s*care\s*magic|hcm|icliniq)[^\n]*\n?|'  
    r'hello\.?\s*|hi\.?\s*|thank\s+you\s+for\s+contacting[^\n]*\n?)',
    re.IGNORECASE
)

def strip_boilerplate(patient_text, doctor_text):
    patient_text = PAT_LEADING.sub('', patient_text).strip()
    doctor_text = DOC_LEADING_WELCOME.sub('', doctor_text).strip()
    doctor_text = DOC_TRAILING.sub('', doctor_text).strip()
    return patient_text, doctor_text

df[['Patient', 'Doctor']] = [
    strip_boilerplate(p, d)
    for p, d in zip(df['Patient'], df['Doctor'])
]
# Drop rows that became empty after stripping
df = df[df['Patient'].str.len() > 20]
df = df[df['Doctor'].str.len() > 30]
log_step('3 — strip boilerplate', df)

In [ ]:
# 2.4 — Length filters (character-based proxy for token length)
# Min: doctor answers shorter than ~80 chars are typically uninformative ("Yes", "See a doctor")
# Max: cap at ~8000 chars (~2048 tokens) to stay within max_seq_length budget

MIN_DOC_CHARS = 80
MAX_COMBINED_CHARS = 8000  # conservative proxy: ~4 chars per token

df = df[df['Doctor'].str.len() >= MIN_DOC_CHARS]
df['combined_len'] = df['Patient'].str.len() + df['Doctor'].str.len()
df = df[df['combined_len'] <= MAX_COMBINED_CHARS]
df = df.drop(columns=['combined_len'])
log_step('4 — length filters', df)

In [ ]:
# 2.5 — Exact deduplication
df = df.drop_duplicates(subset=['Patient', 'Doctor'])
log_step('5 — exact dedup', df)

In [ ]:
# 2.6 — Near-duplicate detection using MinHash LSH
# Medical Q&A scrapes have many paraphrase duplicates that exact dedup misses.
# We hash shingles of the Doctor answer (the informative half) and flag duplicates.
from datasketch import MinHash, MinHashLSH

NUM_PERM = 128   # permutations for MinHash accuracy
THRESHOLD = 0.85 # Jaccard similarity threshold → treat as duplicate

def make_minhash(text, num_perm=NUM_PERM):
    m = MinHash(num_perm=num_perm)
    # 3-character shingles of lowercased, stripped text
    text_norm = re.sub(r'\s+', ' ', text.lower().strip())
    for i in range(len(text_norm) - 2):
        m.update(text_norm[i:i+3].encode('utf8'))
    return m

print(f"Building MinHash signatures for {len(df):,} Doctor answers ...")
df = df.reset_index(drop=True)
minhashes = [make_minhash(doc) for doc in df['Doctor']]

lsh = MinHashLSH(threshold=THRESHOLD, num_perm=NUM_PERM)
to_drop = set()

print("Running LSH query ...")
for i, m in enumerate(minhashes):
    key = str(i)
    result = lsh.query(m)
    if result:          # similar item already indexed → mark as near-dup
        to_drop.add(i)
    else:
        lsh.insert(key, m)

print(f"Near-duplicates identified: {len(to_drop):,}")
df = df.drop(index=list(to_drop)).reset_index(drop=True)
log_step('6 — near-dedup (MinHash)', df)

In [ ]:
# 2.7 — Language filter: keep English only
# Run on the reduced set (faster now that we've deduplicated)
print("Detecting languages (this may take 1–2 min) ...")
df = df.copy()
df['lang'] = df['Doctor'].apply(safe_detect)

lang_dist = df['lang'].value_counts()
print("\nLanguage distribution after cleaning:")
print(lang_dist.head(10))

df = df[df['lang'] == 'en'].drop(columns=['lang'])
log_step('7 — English filter', df)

In [ ]:
# 2.8 — PII scrub: remove obvious emails and phone numbers
# Lightweight regex pass — not a full anonymizer, but removes the most common patterns.
EMAIL_RE = re.compile(r'[\w.+-]+@[\w.-]+\.\w{2,}')
PHONE_RE = re.compile(r'(\+?1[\s.-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}')

def scrub_pii(text):
    text = EMAIL_RE.sub('[EMAIL]', text)
    text = PHONE_RE.sub('[PHONE]', text)
    return text

df['Patient'] = df['Patient'].apply(scrub_pii)
df['Doctor'] = df['Doctor'].apply(scrub_pii)
log_step('8 — PII scrub', df)

In [ ]:
# 2.9 — Sample down to target size
# We keep the full cleaned set saved to Drive but train on a curated 15k.
df_full_clean = df.copy()  # preserved below

df_train_pool = df.sample(n=min(TARGET_ROWS, len(df)), random_state=SEED).reset_index(drop=True)
log_step(f'9 — sample to {TARGET_ROWS:,}', df_train_pool)
print(f"\nFull cleaned set: {len(df_full_clean):,} rows")
print(f"Training pool:    {len(df_train_pool):,} rows")

---
## Phase 3 — Formatting for LoRA
**Why:** The model must train on the **exact** conversational format it will see at inference. Using the tokenizer's own `apply_chat_template` guarantees the special tokens (BOS, EOS, role markers) are placed correctly — a mismatch here silently destroys generation quality.

The system prompt also injects a **medical disclaimer**, which serves two purposes: (1) safety/bias mitigation for the Cyber deliverable; (2) the fine-tuned model will reproduce it at inference, giving users appropriate context.

In [ ]:
# 3.1 — Define system prompt with medical disclaimer
SYSTEM_PROMPT = (
    "You are a helpful medical information assistant. "
    "Provide clear, evidence-based, and cautious guidance. "
    "Always remind users to consult a qualified healthcare professional "
    "for diagnosis, treatment, and medication decisions. "
    "This response is for informational purposes only and does not constitute medical advice."
)

def to_chat_messages(row):
    return [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": row["Patient"].strip()},
        {"role": "assistant", "content": row["Doctor"].strip()},
    ]

df_train_pool['messages'] = [
    to_chat_messages(row) for _, row in df_train_pool.iterrows()
]
print("Chat messages constructed. Sample:")
print(json.dumps(df_train_pool['messages'].iloc[0], indent=2))

In [ ]:
# 3.2 — Apply tokenizer chat template and verify token lengths
# Tokenizer loaded in Phase 1 as `tok`

def format_with_template(messages):
    return tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

df_train_pool['text'] = df_train_pool['messages'].apply(format_with_template)

# Token length check post-template (includes special tokens)
sample_tokens = df_train_pool['text'].sample(500, random_state=SEED).apply(
    lambda t: len(tok.encode(t, truncation=False))
)
over_budget = (sample_tokens > MAX_SEQ_LENGTH).sum()
print(f"Post-template token length (500-sample): median={sample_tokens.median():.0f}, "
      f"p95={sample_tokens.quantile(0.95):.0f}, max={sample_tokens.max()}")
print(f"Rows over {MAX_SEQ_LENGTH} tokens in sample: {over_budget} ({over_budget/5:.1f}%)")
print("\nFormatted example (first 600 chars):")
print(df_train_pool['text'].iloc[0][:600])

In [ ]:
# 3.3 — Train / validation split  (95 / 5)
# Hold-out eval set: 20 fixed questions for Phase 7 side-by-side comparison
from sklearn.model_selection import train_test_split

df_tv, df_eval_fixed = train_test_split(
    df_train_pool, test_size=20, random_state=SEED
)
df_train, df_val = train_test_split(
    df_tv, test_size=0.05, random_state=SEED
)

print(f"Train:     {len(df_train):,} rows")
print(f"Val:       {len(df_val):,} rows")
print(f"Eval hold: {len(df_eval_fixed):,} rows (fixed for Phase 7 comparison)")

---
## Phase 4 — Data Quality Report + Export
**Why:** Two of four required deliverables ("dataset préparé et nettoyé" + "rapport de qualité des données"). The audit log built step-by-step in Phase 2 is the authoritative source — we compile it here and persist everything to Drive.

In [ ]:
# 4.1 — Quality report: funnel table
print("=" * 55)
print("DATA QUALITY REPORT — Medical Dataset Pipeline")
print("=" * 55)

df_audit = pd.DataFrame(audit)
df_audit['rows_dropped'] = df_audit['rows'].shift(1, fill_value=df_audit['rows'].iloc[0]) - df_audit['rows']
df_audit['pct_remaining'] = (df_audit['rows'] / df_audit['rows'].iloc[0] * 100).round(1)
print(df_audit.to_string(index=False))

print(f"\nFinal training pool:       {len(df_train_pool):,}")
print(f"Train split:               {len(df_train):,}")
print(f"Val split:                 {len(df_val):,}")
print(f"Eval hold-out (Phase 7):   {len(df_eval_fixed):,}")

In [ ]:
# 4.2 — Before / after examples (3 samples)
print("\n" + "=" * 55)
print("BEFORE / AFTER EXAMPLES")
print("=" * 55)

# Match cleaned rows back to raw for display
sample_idx = df_train_pool.sample(3, random_state=SEED).index

for idx in sample_idx[:3]:
    clean_row = df_train_pool.loc[idx]
    print(f"\n{'─'*50}")
    print(f"[CLEANED Patient]:  {clean_row['Patient'][:300]}")
    print(f"[CLEANED Doctor]:   {clean_row['Doctor'][:400]}")

In [ ]:
# 4.3 — Post-cleaning length distribution plot
df_train_pool['doc_len_clean'] = df_train_pool['Doctor'].str.len()
df_train_pool['pat_len_clean'] = df_train_pool['Patient'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col, label, color in zip(
    axes,
    ['pat_len_clean', 'doc_len_clean'],
    ['Patient (after cleaning)', 'Doctor (after cleaning)'],
    ['mediumseagreen', 'cornflowerblue']
):
    ax.hist(df_train_pool[col], bins=60, color=color, edgecolor='white', linewidth=0.4)
    ax.set_title(f'{label}')
    ax.set_xlabel('Characters'); ax.set_ylabel('Count')

plt.suptitle(f'Cleaned dataset — {len(df_train_pool):,} rows', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/reports/cleaned_length_distributions.png', dpi=120)
plt.show()

In [ ]:
# 4.4 — Export all datasets to Drive

# Full cleaned parquet (for team)
full_clean_path = f'{DRIVE_BASE}/data/medical_clean_full.parquet'
df_full_clean[['Patient', 'Doctor']].to_parquet(full_clean_path, index=False)
print(f"Saved full cleaned dataset:  {full_clean_path}  ({len(df_full_clean):,} rows)")

# Training pool parquet
pool_path = f'{DRIVE_BASE}/data/medical_train_pool_{TARGET_ROWS}.parquet'
df_train_pool[['Patient', 'Doctor']].to_parquet(pool_path, index=False)
print(f"Saved training pool:         {pool_path}  ({len(df_train_pool):,} rows)")

# JSONL files: chat-formatted (for training notebook)
def save_jsonl(df, path):
    with open(path, 'w', encoding='utf-8') as f:
        for _, row in df.iterrows():
            f.write(json.dumps({'text': row['text']}, ensure_ascii=False) + '\n')
    print(f"Saved JSONL:                 {path}  ({len(df):,} rows)")

save_jsonl(df_train, f'{DRIVE_BASE}/data/train.jsonl')
save_jsonl(df_val,   f'{DRIVE_BASE}/data/val.jsonl')

# Eval hold-out: save raw (Patient / Doctor) for side-by-side comparison in Phase 7
eval_path = f'{DRIVE_BASE}/data/eval_holdout.jsonl'
df_eval_fixed[['Patient', 'Doctor']].to_json(eval_path, orient='records', lines=True, force_ascii=False)
print(f"Saved eval hold-out:         {eval_path}  ({len(df_eval_fixed):,} rows)")

# Save audit funnel as CSV
audit_path = f'{DRIVE_BASE}/reports/data_quality_report.csv'
df_audit.to_csv(audit_path, index=False)
print(f"Saved quality report CSV:    {audit_path}")

In [ ]:
# 4.5 — Print summary banner
print()
print("╔══════════════════════════════════════════════════════╗")
print("║         PHASE 0–4 COMPLETE — DATA DELIVERABLES       ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  Raw rows:             {df_raw.shape[0]:>7,}                       ║")
print(f"║  Full cleaned rows:    {len(df_full_clean):>7,}                       ║")
print(f"║  Training pool (15k):  {len(df_train_pool):>7,}                       ║")
print(f"║  Train / Val / Eval:  {len(df_train):>6,} / {len(df_val):>4,} / {len(df_eval_fixed):>2,}              ║")
print("╠══════════════════════════════════════════════════════╣")
print("║  Files on Drive:                                     ║")
print("║    data/medical_clean_full.parquet                   ║")
print(f"║    data/medical_train_pool_{TARGET_ROWS}.parquet         ║")
print("║    data/train.jsonl  (chat-formatted, for training)  ║")
print("║    data/val.jsonl                                    ║")
print("║    data/eval_holdout.jsonl                           ║")
print("║    reports/data_quality_report.csv                   ║")
print("║    reports/char_length_distributions.png             ║")
print("║    reports/token_length_distribution.png             ║")
print("║    reports/cleaned_length_distributions.png          ║")
print("╠══════════════════════════════════════════════════════╣")
print("║  NEXT → medical_training_pipeline.ipynb              ║")
print("║         (Phases 5–8: validation, LoRA, eval, export) ║")
print("╚══════════════════════════════════════════════════════╝")